# Day 2 — Runtime Architecture

**What you will learn:**
- The three components of a Spark cluster
- How a query becomes Jobs → Stages → Tasks
- Narrow vs wide transformations — why it matters
- Deployment modes: local, client, cluster
- How Spark recovers from failures

**Exam relevance:** Architecture (20%) — execution hierarchy and deployment modes are tested directly.

## 1. Cluster Components

```
┌─────────────────────────────────────────────────┐
│                  SPARK CLUSTER                  │
│                                                 │
│  ┌──────────────┐       ┌───────────────────┐  │
│  │    DRIVER    │◄─────►│  CLUSTER MANAGER  │  │
│  │              │       │  (YARN/K8s/DBR)   │  │
│  │ SparkContext │       └───────────────────┘  │
│  │ Scheduler    │               │               │
│  │ DAG Builder  │       ┌───────┴───────┐       │
│  └──────────────┘       ▼               ▼       │
│                   ┌──────────┐  ┌──────────┐   │
│                   │EXECUTOR 1│  │EXECUTOR 2│   │
│                   │ Task Task│  │ Task Task│   │
│                   │ [cache]  │  │ [cache]  │   │
│                   └──────────┘  └──────────┘   │
└─────────────────────────────────────────────────┘
```

| Component | Role |
|---|---|
| **Driver** | Your notebook / application. Builds the execution plan, schedules tasks, collects results. |
| **Cluster Manager** | Allocates resources (workers). Options: Databricks, YARN, Kubernetes, Mesos, Standalone. |
| **Executors** | JVM processes on worker nodes. Run tasks, store cached data. Created at app start, live for its duration. |

In [ ]:
# sparkContext JVM access is not available on Databricks serverless
try:
    sc = spark.sparkContext
    print(f"Master             : {sc.master}")
    print(f"Default parallelism: {sc.defaultParallelism}")  # total cores across all executors
    print(f"App name           : {sc.appName}")
except Exception:
    print("Note: sparkContext JVM attributes unavailable on serverless compute.")
    print("On a classic cluster these return the master URL, total cores, and app name.")
    print(f"Spark version: {spark.version}  — session is active and working.")

## 2. Jobs → Stages → Tasks

When you call an **action** (`.show()`, `.count()`, `.write()`), Spark creates a **Job**.  
Each job is broken into **Stages** at shuffle boundaries.  
Each stage is split into **Tasks** — one task per partition.

```
ACTION called
    └── JOB
         ├── STAGE 1  (narrow transformations — no shuffle)
         │     ├── Task (partition 0)
         │     ├── Task (partition 1)
         │     └── Task (partition N)
         └── STAGE 2  (after shuffle boundary)
               ├── Task (partition 0)
               └── Task (partition 1)
```

> **Rule:** Every shuffle creates a new stage boundary.

In [ ]:
data = [(i, f"user_{i % 5}", i * 10) for i in range(1, 21)]
df = spark.createDataFrame(data, ["id", "user", "amount"])

# getNumPartitions via RDD is not available on serverless — use try/except
try:
    print(f"Partitions: {df.rdd.getNumPartitions()}")
except Exception:
    print("Note: .rdd.getNumPartitions() not available on serverless.")
    print("On a classic cluster this returns the number of data partitions.")

# This groupBy triggers a shuffle → 2 stages — works on all compute types
result = df.groupBy("user").sum("amount")
result.show()
# Open the Spark UI (cluster → Spark UI tab) to see the 2 stages

## 3. Narrow vs Wide Transformations

This is a core exam concept — know the difference cold.

| | Narrow | Wide |
|---|---|---|
| **Data movement** | None — each partition is processed independently | Shuffle — data moves across the network |
| **Stage boundary** | No | Yes — creates a new stage |
| **Examples** | `filter`, `select`, `withColumn`, `map`, `union` | `groupBy`, `join`, `distinct`, `repartition`, `orderBy` |
| **Performance** | Fast | Slower (network I/O) |

```
NARROW (filter)              WIDE (groupBy)

P0 ──filter──► P0'          P0 ─┐
P1 ──filter──► P1'          P1 ─┼──shuffle──► P0' (grouped)
P2 ──filter──► P2'          P2 ─┘             P1' (grouped)

No data crosses partitions   Data redistributed across network
```

In [ ]:
from pyspark.sql.functions import col

# Narrow — filter stays within each partition, no shuffle
narrow = df.filter(col("amount") > 50).select("id", "amount")
narrow.explain()  # 1 stage expected

# Wide — groupBy requires data from all partitions to be redistributed
wide = df.groupBy("user").count()
wide.explain()  # 2 stages: scan+shuffle write, then shuffle read+aggregate

## 4. Deployment Modes

| Mode | Driver location | Typical use |
|---|---|---|
| **local** | Same machine as executor | Local dev/testing — `local[*]` uses all cores |
| **client** | Machine submitting the job | Interactive notebooks (Databricks, Jupyter) |
| **cluster** | A worker node in the cluster | Production batch jobs (`spark-submit`) |

> **Exam tip:** In **client mode**, if the submitting machine fails, the Driver dies and the job fails.  
> In **cluster mode**, the Driver runs inside the cluster — the submitting machine can disconnect safely.

In Databricks, you always run in **client mode** from the notebook.

In [ ]:
# Verify current mode and config
print(spark.conf.get("spark.submit.deployMode", "client (default)"))
print(spark.conf.get("spark.executor.memory", "not set"))
print(spark.conf.get("spark.executor.cores", "not set"))

## 5. Fault Tolerance via Lineage

Spark does **not** replicate data between steps. Instead it remembers the **lineage** — the full sequence of transformations that produced each partition.

If an executor dies mid-job:
1. The Cluster Manager detects the failure
2. Spark re-schedules only the **failed tasks** on a healthy executor
3. The lineage is replayed from the last checkpoint or from the source

> **Exam tip:** Spark's fault tolerance model is **lineage-based recomputation**, not replication. RDDs store the recipe, not copies of the data.

In [ ]:
# toDebugString shows RDD lineage — not available on serverless
try:
    rdd = df.filter(col("amount") > 50).select("user").rdd
    print(rdd.toDebugString().decode())
except Exception:
    print("Note: .rdd and toDebugString() not available on serverless compute.")
    print("On a classic cluster this prints the full lineage chain, e.g.:")
    print("  MapPartitionsRDD -> FilteredRDD -> MapPartitionsRDD -> ParallelCollectionRDD")
    print()
    print("The concept still applies on serverless — use explain() to see the logical lineage:")
    df.filter(col("amount") > 50).select("user").explain()

---

## Day 2 Checklist

- [ ] Can draw the Driver / Cluster Manager / Executor triangle from memory
- [ ] Understand: Action → Job → Stages → Tasks
- [ ] Know which transformations are narrow and which are wide
- [ ] Can explain why groupBy creates a stage boundary (shuffle)
- [ ] Know the 3 deployment modes and when each is used
- [ ] Understand fault tolerance via lineage recomputation

**Next:** Day 3 — Lazy Evaluation & the Catalyst Optimizer

---

## Interactive Quiz — Day 2 Architecture

Five topic areas, three questions each. Pick a topic, answer the questions, then move on. Run the cell below to launch the quiz.

In [ ]:
displayHTML("""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Spark Fundamentals Quiz</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #fff; color: #1a1a1a; padding: 1.5rem; max-width: 720px; }
  h1 { font-size: 18px; font-weight: 500; margin-bottom: 1.25rem; color: #1a1a1a; }
  .topic-bar { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 1.5rem; }
  .topic-btn { padding: 7px 13px; border: 1px solid #ccc; border-radius: 8px; background: #fff; color: #1a1a1a; cursor: pointer; font-size: 13px; font-family: inherit; }
  .topic-btn:hover { background: #f5f5f5; }
  .topic-btn.active { background: #e8f0fe; color: #1a56db; border-color: #93b4f8; }
  .meta { display: flex; justify-content: space-between; margin-bottom: 1rem; }
  .meta span { font-size: 13px; color: #666; }
  .question { font-size: 15px; font-weight: 500; margin-bottom: 1rem; line-height: 1.5; }
  .opt-btn { width: 100%; text-align: left; padding: 10px 14px; margin: 6px 0; border: 1px solid #ddd; border-radius: 8px; background: #fff; color: #1a1a1a; cursor: pointer; font-size: 14px; font-family: inherit; line-height: 1.4; }
  .opt-btn:hover { background: #f5f5f5; }
  .opt-btn.correct { background: #ecfdf5 !important; border-color: #6ee7b7 !important; color: #065f46 !important; pointer-events: none; }
  .opt-btn.wrong { background: #fef2f2 !important; border-color: #fca5a5 !important; color: #991b1b !important; pointer-events: none; }
  .opt-btn.locked { pointer-events: none; }
  .explanation { padding: 10px 14px; border-left: 3px solid #93b4f8; font-size: 14px; color: #444; margin-top: 1rem; line-height: 1.6; background: #f8faff; border-radius: 0 6px 6px 0; }
  .nav { margin-top: 1rem; }
  .score-line { font-size: 14px; color: #555; margin-bottom: 8px; }
  .placeholder { font-size: 15px; color: #888; }
</style>
</head>
<body>

<h1>Spark fundamentals quiz</h1>
<div class="topic-bar" id="topic-bar"></div>
<div id="quiz-area"><p class="placeholder">Pick a topic above to start.</p></div>

<script>
const topics = [
  {
    id: "arch", label: "Driver / Cluster / Executor",
    questions: [
      { q: "Which component breaks a Spark application into tasks and schedules them?", opts: ["Cluster Manager","Driver","Executor","Worker Node"], ans: 1, exp: "The Driver runs your main() function, creates the SparkContext, builds the DAG, and schedules tasks onto Executors." },
      { q: "What does the Cluster Manager do?", opts: ["Runs transformations on data","Allocates resources (CPU/memory) across the cluster","Stores the lineage graph","Serializes DataFrames"], ans: 1, exp: "The Cluster Manager (YARN, Mesos, Kubernetes, Standalone) negotiates resources. It doesn't touch your data." },
      { q: "Where does actual data processing happen?", opts: ["Driver JVM","Cluster Manager","Executors","SparkContext"], ans: 2, exp: "Executors are JVM processes on Worker nodes. They run tasks, cache data, and report results back to the Driver." }
    ]
  },
  {
    id: "jobstages", label: "Action → Job → Stages → Tasks",
    questions: [
      { q: "What triggers Spark to actually execute a computation?", opts: ["A transformation like map()","An action like count() or show()","Creating a DataFrame","Calling explain()"], ans: 1, exp: "Transformations are lazy — they just build a plan. An action like collect(), count(), show(), or write() triggers execution." },
      { q: "What creates a stage boundary in Spark?", opts: ["Any transformation","A narrow transformation","A shuffle (wide transformation)","Calling cache()"], ans: 2, exp: "Shuffles require data to move between partitions across the network — Spark must complete Stage N before starting Stage N+1." },
      { q: "How does a Task relate to a Stage?", opts: ["One Task per Job","One Task per Stage","One Task per partition within a Stage","One Task per Executor"], ans: 2, exp: "A Stage is split into Tasks — one per partition. 200 partitions → 200 Tasks, each running on one Executor slot." }
    ]
  },
  {
    id: "transformations", label: "Narrow vs Wide",
    questions: [
      { q: "Which of these is a narrow transformation?", opts: ["groupBy()","join() on two large tables","filter()","distinct()"], ans: 2, exp: "filter() keeps each row in its current partition — no data moves. groupBy, join, and distinct all require shuffles." },
      { q: "Why is groupBy() a wide transformation?", opts: ["It creates a new DataFrame","Rows with the same key may live in different partitions and must be co-located","It always reads from disk","It uses more memory"], ans: 1, exp: "To aggregate by 'user', all rows for user_1 must land on the same executor. Since they're scattered, a shuffle is needed." },
      { q: "Which set contains only narrow transformations?", opts: ["map, filter, select","groupBy, join, repartition","distinct, orderBy, join","map, groupBy, filter"], ans: 0, exp: "map, filter, and select operate partition-by-partition with no cross-partition data movement. All others trigger shuffles." }
    ]
  },
  {
    id: "deploy", label: "Deployment modes",
    questions: [
      { q: "In client mode, where does the Driver run?", opts: ["On a worker node inside the cluster","On the machine that submitted the job","On the Cluster Manager","On the first Executor"], ans: 1, exp: "Client mode: Driver stays on your local machine. Good for interactive notebooks. Bad for long jobs — if your machine dies, the job dies." },
      { q: "In cluster mode, what happens to the Driver?", opts: ["It runs on the submitting machine","It's deployed inside the cluster on a worker node","It merges with the Cluster Manager","It's not used"], ans: 1, exp: "Cluster mode: Driver runs inside the cluster. The submitting process exits after launch. Best for production batch jobs." },
      { q: "When would you use local mode?", opts: ["Production ETL pipelines","Multi-node distributed jobs","Development, testing, and learning on a single machine","When the cluster is down for maintenance"], ans: 2, exp: "local[*] runs Driver and Executors in the same JVM process. No cluster needed — great for development and unit tests." }
    ]
  },
  {
    id: "fault", label: "Fault tolerance",
    questions: [
      { q: "How does Spark recover from a lost partition without rerunning everything?", opts: ["Restores from a checkpoint on HDFS","Uses lineage to recompute only the lost partition","Asks another Executor to copy its partition","Re-reads all source data from scratch"], ans: 1, exp: "Spark tracks the DAG of transformations (lineage). If a partition is lost, it replays only the transformations needed to rebuild it." },
      { q: "What is the lineage graph?", opts: ["A log of completed tasks","A record of which Executors ran which tasks","A DAG showing how each partition was derived from transformations","The physical execution plan after optimization"], ans: 2, exp: "The lineage (DAG) records every transformation applied to produce each partition — the recipe Spark uses to rebuild lost data." },
      { q: "When would you explicitly checkpoint a DataFrame?", opts: ["Before every action","When the lineage chain becomes very long, to truncate recomputation cost","Only with streaming jobs","When switching from client to cluster mode"], ans: 1, exp: "After many chained transformations, recomputing from scratch gets expensive. Checkpointing saves a snapshot and truncates the lineage." }
    ]
  }
];

const scores = {};
topics.forEach(function(t) { scores[t.id] = { correct: 0, total: 0 }; });
let activeTopic = null;
let activeQ = 0;
let answered = false;

function renderTopicBar() {
  const bar = document.getElementById("topic-bar");
  bar.innerHTML = "";
  topics.forEach(function(t) {
    const s = scores[t.id];
    const btn = document.createElement("button");
    btn.className = "topic-btn" + (activeTopic === t.id ? " active" : "");
    btn.textContent = t.label + (s.total > 0 ? " " + s.correct + "/" + s.total : "");
    btn.addEventListener("click", function() {
      activeTopic = t.id; activeQ = 0; answered = false; renderAll();
    });
    bar.appendChild(btn);
  });
}

function renderQuestion() {
  const area = document.getElementById("quiz-area");
  if (!activeTopic) {
    area.innerHTML = "<p class='placeholder'>Pick a topic above to start.</p>";
    return;
  }
  const topic = topics.find(function(t) { return t.id === activeTopic; });
  const q = topic.questions[activeQ];
  area.innerHTML =
    "<div class='meta'><span>" + topic.label + "</span><span>" + (activeQ+1) + " / " + topic.questions.length + "</span></div>" +
    "<p class='question'>" + q.q + "</p>" +
    "<div id='opts-wrap'></div>" +
    "<div id='exp-wrap'></div>" +
    "<div id='nav-wrap' class='nav'></div>";

  const wrap = document.getElementById("opts-wrap");
  q.opts.forEach(function(opt, i) {
    const btn = document.createElement("button");
    btn.className = "opt-btn";
    btn.textContent = opt;
    btn.addEventListener("click", function() { handleAnswer(i); });
    wrap.appendChild(btn);
  });
}

function handleAnswer(chosen) {
  if (answered) return;
  answered = true;
  const topic = topics.find(function(t) { return t.id === activeTopic; });
  const q = topic.questions[activeQ];
  scores[activeTopic].total++;
  if (chosen === q.ans) scores[activeTopic].correct++;

  document.querySelectorAll(".opt-btn").forEach(function(b, i) {
    b.classList.add("locked");
    if (i === q.ans) b.classList.add("correct");
    else if (i === chosen) b.classList.add("wrong");
  });

  document.getElementById("exp-wrap").innerHTML =
    "<div class='explanation'>" + q.exp + "</div>";

  const nav = document.getElementById("nav-wrap");
  if (activeQ < topic.questions.length - 1) {
    const nb = document.createElement("button");
    nb.className = "topic-btn";
    nb.textContent = "Next question →";
    nb.addEventListener("click", function() { activeQ++; answered = false; renderAll(); });
    nav.appendChild(nb);
  } else {
    const s = scores[activeTopic];
    nav.innerHTML = "<p class='score-line'>Topic done — score: <strong>" + s.correct + "/" + s.total + "</strong></p>";
    const rb = document.createElement("button");
    rb.className = "topic-btn";
    rb.textContent = "Retry topic";
    rb.addEventListener("click", function() { scores[activeTopic]={correct:0,total:0}; activeQ=0; answered=false; renderAll(); });
    nav.appendChild(rb);
  }
  renderTopicBar();
}

function renderAll() { renderTopicBar(); renderQuestion(); }
renderAll();
</script>
</body>
</html>""")